# WI-91 探针夹具的**变体**：摘掉定义那一格（主夹具的 cell 1）

配套 `99-probe-scala.ipynb` 用，两份**都要跑**（全局约束里 2026-10-10 由 Task 1 实测补进来的那一条）。

## 为什么非要有这一份

`nbconvert --execute` 是**线性执行一整本** notebook 的。于是「把整本一次性编译、再按 cell 切输出」的那种
**假 kernel**（本质是编译器不是 REPL）能同时交出主夹具那三条判据的读数：它确实能让第二个 cell 拿到 42，
确实走 stream 通道，也确实起得来 SparkSession。摘掉定义那一格之后它没有退路 ——
这里引用的 `doubled` 在**它自己的历史里从没被绑定过**。

## 这一份的判据是**反的**

| 读数 | 期望 | 为什么 |
| --- | --- | --- |
| `stateSurvived` | **false** | 没有任何地方定义过 `doubled`，`alive=42` 不该出现 |
| `errors` | **非空**，第一条落在引用 `doubled` 的那个 code cell（0 基下标 1） | 真 REPL 在这里报 `not found: value doubled` |
| `sparkSession` | 不参与判定（可以是 true） | Spark 那一格不依赖 `doubled`，它绿不说明任何事 |

⇒ 一句话：**主夹具绿 + 这一份按上表红**，两条一起才叫「跨 cell 状态活着，而且不是整本一次性编译」。
单看这一份的红什么都证明不了（一条连 Scala 都不认的 kernel 也红），单看主夹具的绿又证明不了交互性。

## 已知边界

- 若某条 kernel 把「未定义符号」吞成一条 warning 而不是 `output_type=error`，`errors` 这一栏会是空的；
  但 `alive=42` 依然不可能出现 ⇒ `stateSurvived=false` 仍然是正确读数，判据不靠 `errors` 那一栏成立。
- 执行器（`scalaKernelProbe.ts`）在调用方显式给 `notebookPath` 时**跳过**夹具形状自检与
  `language_info` 身份判据（语义已经换了）。所以这一份的 kernel 身份要么看主夹具那次的读数，
  要么在本文件里另判 —— 别把「这一份没报身份问题」读成「身份没问题」。

In [ ]:
println(s"alive=$doubled")

In [ ]:
val spark = org.apache.spark.sql.SparkSession.builder().appName("a2-probe-nodef").getOrCreate()
println(s"rows=${spark.range(6).count()}")   // 6，不是 15：这一条只数行数
spark.stop()